# Client A — Combined Working Notebook

**Who wrote what:** Vicky’s shared data preparation comes first. **Zac’s Client A strategy is the final three code cells**, under the heading below. This is a separate working copy, not the final group submission.


In [ ]:
from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## Part 1 — Shared data preparation (Vicky)

Reads and screens the ticker list, downloads prices and volume, converts to CAD, and provides the signals used by both clients.


In [ ]:
import time  # not in the given import cell; needed for time.sleep

GROUP_NUMBER = "6"        
TICKER_FILE = "Tickers.csv"

PURCHASE_DATE = pd.Timestamp("2026-11-27")
# Competition holding window (used to find earnings reports during the competition)
HOLD_START = pd.Timestamp("2026-11-27 16:00", tz="America/New_York")
HOLD_END   = pd.Timestamp("2026-12-04 16:00", tz="America/New_York")

# Price history to download: must start before the volume window and cover a year of signals
HISTORY_START = pd.Timestamp("2024-10-01")

# Volume rule
VOLUME_START = pd.Timestamp("2025-10-01")
VOLUME_END   = pd.Timestamp("2026-09-30")
MIN_AVG_VOLUME = 5000
MIN_TRADING_DAYS_IN_MONTH = 18

# Client mandates
CLIENT_BUDGETS = {"A": 1_000_000, "B": 750_000}       # CAD
CLIENT_N_STOCKS = {"A": 12, "B": 20}                   # our chosen portfolio sizes
MIN_STOCKS, MAX_STOCKS = 10, 20
MAX_STOCK_WEIGHT = 0.15
MAX_SECTOR_WEIGHT = 0.40

# Transaction fee: the smaller of US$2.15 and US$0.001 per share, for every stock
MAX_FEE_USD = 2.15
FEE_PER_SHARE_USD = 0.001
SAFETY_BUFFER_CAD = 0.01     # leave one cent unspent so rounding can never exceed the budget

# Eligibility (yfinance codes)
ALLOWED_QUOTE_TYPES = ["EQUITY"]
ALLOWED_CURRENCIES  = ["CAD", "USD"]
US_EXCHANGES = ["NMS", "NGM", "NCM", "NYQ", "ASE", "PCX", "BTS"]  # Nasdaq tiers, NYSE, NYSE American, NYSE Arca, Cboe
CA_EXCHANGES = ["TOR", "VAN", "NEO", "CNQ"]                       # TSX, TSX Venture, Cboe Canada, CSE
# OTC / pink-sheet codes are deliberately excluded: they are not exchange listings.
MAX_STALE_DAYS = 5           # last price older than this before the purchase date = not trading

# Strategy parameters
TRADING_DAYS_PER_YEAR = 252
LOOKBACK_DAYS = 252          # one year: Client B volatility and the equal-weight comparison period
MOMENTUM_DAYS = 126          # about six months
SHORT_MOMENTUM_DAYS = 20     # about one month
TREND_DAYS = 50              # 50-day moving average trend filter
CATALYST_BONUS = 0.10        # score bonus for Client A stocks with earnings in the window and positive momentum

BENCHMARK_TICKERS = {"US": "^GSPC", "CA": "^GSPTSE"}  # S&P 500 and S&P/TSX Composite
FX_TICKER = "CAD=X"          # CAD per 1 USD
PAUSE_SECONDS = 0.2          # pause between per-ticker requests to avoid rate limits





## read the ticker file and collect company information

def read_tickers(path):
    """Read the ticker file (one ticker per row, no header) in order to return a clean, de-duplicated list."""
    raw = pd.read_csv(path, header=None) ## pd haven't touched the data
    tickers = (raw.iloc[:, 0].dropna().astype(str)
               .str.strip().str.upper())
    tickers = tickers[tickers != ""].drop_duplicates()
    return list(tickers)


def get_stock_info(tickers):
    """Looking up quote type, exchange, currency and sector for each ticker.
    Tickers that yfinance cannot find will result in empty values (they fail the checks later)."""
    rows = []
    for ticker in tickers:
        try:
            info = yf.Ticker(ticker).info
        except Exception:
            info = {}
        rows.append({
            "Ticker": ticker,
            "SecurityType": info.get("quoteType"),
            "Exchange": info.get("exchange"),
            "Currency": info.get("currency"),
            "Sector": info.get("sector"),
            "Name": info.get("longName") or info.get("shortName"),
        })
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")


def market_of(exchange):
    """Return 'US', 'CA' or None depending on where the stock is listed."""
    if pd.isna(exchange):
        return None
    if exchange in US_EXCHANGES:
        return "US"
    if exchange in CA_EXCHANGES:
        return "CA"
    return None



## download price and volume history
#### Use auto_adjust=False and Close column


def download_history(tickers, start, end):
    """Download daily closes and volumes
       return into two DFs (dates x tickers)"""

    data = yf.download(tickers, start=start, end=end + pd.Timedelta(days=1),
                       auto_adjust=False, progress=False, group_by="column")

    closes = data["Close"]
    volumes = data["Volume"]
    # With one ticker yfinance can return a Series; make it a DataFrame either way
    if isinstance(closes, pd.Series):
        closes = closes.to_frame(tickers[0])
        volumes = volumes.to_frame(tickers[0])
    closes.index = pd.to_datetime(closes.index).tz_localize(None)
    volumes.index = pd.to_datetime(volumes.index).tz_localize(None)
    return closes, volumes 



def get_fx_rate(index, start, end):
    """CAD per USD, lined up with the stock dates (forward-filled over holidays)."""
    fx = yf.download(FX_TICKER, start=start, end=end + pd.Timedelta(days=1),
                     auto_adjust=False, progress=False)["Close"]
    if isinstance(fx, pd.DataFrame):
        fx = fx.iloc[:, 0]
    fx.index = pd.to_datetime(fx.index).tz_localize(None)
    return fx.reindex(fx.index.union(index)).ffill().reindex(index).rename("USDCAD")



def convert_to_cad(closes_local, currencies, fx):
    """Multiply USD-quoted columns by the daily exchange rate; CAD columns stay as they are."""
    prices_cad = closes_local.copy()
    for ticker in closes_local.columns:
        if currencies.get(ticker) == "USD":
            prices_cad[ticker] = closes_local[ticker] * fx
    return prices_cad




## Volume Rule
## A "trading day" is a day the stock actually has a volume record. Because TSX and US holidays differ, each stock is counted on its own calendar.
## The average is taken over all remaining days (not an average of monthly averages).


def average_volume(volume_series, start, end, min_days):
    """Average daily volume over [start, end], excluding months with fewer than min_days trading days."""
    v = volume_series.loc[start:end].dropna()
    if v.empty:
        return np.nan
    month = v.index.to_period("M")
    days_per_month = v.groupby(month).count()
    full_months = days_per_month[days_per_month >= min_days].index
    v = v[month.isin(full_months)]
    return v.mean() if not v.empty else np.nan


## Eligibility check + load_stock_data()
def check_eligibility(info, closes_local, volumes, as_of):
    """Add AvgVolume, Eligible and Reason columns. Every failed rule is listed in Reason."""
    result = info.copy()
    result["Market"] = result["Exchange"].apply(market_of)
    result["AvgVolume"] = np.nan
    result["Eligible"] = False
    result["Reason"] = ""

    for ticker in result.index:
        problems = []
        row = result.loc[ticker]

        if pd.isna(row["SecurityType"]):
            # yfinance found nothing: report one clear reason and skip the other checks
            result.loc[ticker, "Reason"] = "invalid ticker (no data found)"
            continue
        if row["SecurityType"] not in ALLOWED_QUOTE_TYPES:
            problems.append(f"not a company stock ({row['SecurityType']})")
        if pd.isna(row["Market"]):
            problems.append(f"not listed on a Canadian/US exchange ({row['Exchange']})")
        if row["Currency"] not in ALLOWED_CURRENCIES:
            problems.append(f"currency not CAD/USD ({row['Currency']})")

        if ticker in closes_local.columns:
            prices = closes_local[ticker].dropna()
            last_date = prices.index.max() if not prices.empty else None
            check_date = min(as_of, pd.Timestamp.today().normalize())
            if last_date is None or (check_date - last_date).days > MAX_STALE_DAYS:
                problems.append("no recent price (possibly delisted)")
            avg_vol = average_volume(volumes[ticker], VOLUME_START, VOLUME_END,
                                     MIN_TRADING_DAYS_IN_MONTH)
            result.loc[ticker, "AvgVolume"] = avg_vol
            if pd.isna(avg_vol) or avg_vol < MIN_AVG_VOLUME:
                problems.append(f"average volume below {MIN_AVG_VOLUME:,}")
        else:
            problems.append("no price data")

        result.loc[ticker, "Eligible"] = len(problems) == 0
        result.loc[ticker, "Reason"] = "; ".join(problems)
    return result


def load_stock_data(path, as_of=PURCHASE_DATE):
    """Read tickers, check eligibility and download history.
    Returns a dictionary with the screening table and all historical tables."""
    tickers = read_tickers(path)
    print(f"Read {len(tickers)} unique tickers from {path}")

    info = get_stock_info(tickers)
    found = [t for t in tickers if pd.notna(info.loc[t, "SecurityType"])]
    closes_local, volumes = download_history(found, HISTORY_START, as_of)

    screening = check_eligibility(info, closes_local, volumes, as_of)
    eligible = list(screening.index[screening["Eligible"]])

    # Keep only eligible stocks in the price tables
    closes_local = closes_local[eligible].loc[:as_of]
    volumes = volumes[eligible].loc[:as_of]

    fx = get_fx_rate(closes_local.index, HISTORY_START, as_of)
    prices_cad = convert_to_cad(closes_local, screening["Currency"].to_dict(), fx)

    print(f"{len(eligible)} eligible, {len(tickers) - len(eligible)} excluded")
    return {
        "screening": screening,
        "eligible": eligible,
        "closes_local": closes_local,
        "volumes": volumes,
        "fx": fx,
        "prices_cad": prices_cad,
    }


## benchmarks in CAD

## Each stock is compared with home markets

def get_benchmarks_cad(index, fx, start, end):
    """Return a DataFrame with columns 'US' and 'CA', both in CAD, lined up with the stock dates."""
    symbols = list(BENCHMARK_TICKERS.values())
    closes, _ = download_history(symbols, start, end)
    closes = closes.reindex(closes.index.union(index)).ffill().reindex(index)
    benchmark_cad = pd.DataFrame(index=index)
    benchmark_cad["US"] = closes[BENCHMARK_TICKERS["US"]] * fx   # S&P 500 is in USD
    benchmark_cad["CA"] = closes[BENCHMARK_TICKERS["CA"]]        # TSX is already in CAD
    return benchmark_cad





## Catalysts
def earnings_in_window(timestamps, start=HOLD_START, end=HOLD_END):
    """Return the first earnings timestamp inside the window, or NaT."""
    for ts in sorted(timestamps):
        ts = pd.Timestamp(ts)
        if ts.tzinfo is None:
            ts = ts.tz_localize("America/New_York")
        else:
            ts = ts.tz_convert("America/New_York")
        if ts.hour == 0 and ts.minute == 0:      # time unknown: compare dates only
            if start.date() < ts.date() <= end.date():
                return ts
        elif start < ts < end:
            return ts
    return pd.NaT


def get_earnings_timestamps(ticker):
    """Collect earnings timestamps from get_earnings_dates(), falling back to .calendar."""
    stock = yf.Ticker(ticker)
    timestamps = []
    try:
        dates = stock.get_earnings_dates(limit=8)
        if dates is not None and len(dates) > 0:
            timestamps += list(dates.index)
    except Exception:
        pass
    if not timestamps:
        try:
            calendar = stock.calendar or {}
            timestamps += list(calendar.get("Earnings Date", []))
        except Exception:
            pass
    return timestamps


def get_catalysts(tickers):
    """Return a DataFrame with EarningsDate and CatalystStatus for each ticker."""
    rows = []
    for ticker in tickers:
        timestamps = get_earnings_timestamps(ticker)
        hit = earnings_in_window(timestamps)
        if pd.notna(hit):
            status = "Event"
        elif not timestamps:
            status = "Unknown"
        # Only say "NoEvent" if we can see an upcoming date (otherwise the data may just be stale)
        elif max(pd.Timestamp(t).tz_localize(None) if pd.Timestamp(t).tzinfo is None
                 else pd.Timestamp(t).tz_convert(None) for t in timestamps) > HOLD_START.tz_convert(None):
            status = "NoEvent"
        else:
            status = "Unknown"
        rows.append({"Ticker": ticker, "EarningsDate": hit, "CatalystStatus": status})
        time.sleep(PAUSE_SECONDS)
    return pd.DataFrame(rows).set_index("Ticker")



## Volatility

def window_return(prices, days):
    """Return over the last `days` trading days (prices must have no gaps)."""
    if len(prices) <= days:
        return np.nan
    return prices.iloc[-1] / prices.iloc[-1 - days] - 1


def calculate_signals(prices_cad, volumes, benchmark_cad, markets, as_of=PURCHASE_DATE):
    """Return a DataFrame of signals (one row per ticker), calculated as of `as_of`."""
    prices_cad = prices_cad.loc[:as_of]
    volumes = volumes.loc[:as_of]
    benchmark_cad = benchmark_cad.loc[:as_of]

    # Benchmark 20-day returns, using the benchmark's own trading days
    bench_return20 = {m: window_return(benchmark_cad[m].dropna(), 20) for m in benchmark_cad.columns}

    rows = []
    for ticker in prices_cad.columns:
        prices = prices_cad[ticker].dropna()
        daily_returns = prices.pct_change().dropna()
        vol = volumes[ticker].dropna()

        vol20 = daily_returns.tail(20).std() if len(daily_returns) >= 20 else np.nan
        vol60 = daily_returns.tail(60).std() if len(daily_returns) >= 60 else np.nan
        return20 = window_return(prices, 20)
        ma20 = prices.tail(20).mean() if len(prices) >= 20 else np.nan
        avg_vol_60 = vol.tail(60).mean() if len(vol) >= 60 else np.nan

        rows.append({
            "Ticker": ticker,
            "Vol20": vol20,
            "Vol60": vol60,
            "VolRatio": vol20 / vol60 if vol60 else np.nan,
            "Return5": window_return(prices, 5),
            "Return20": return20,
            "PriceVsMA20": prices.iloc[-1] / ma20 - 1 if ma20 else np.nan,
            "RelativeMomentum": return20 - bench_return20.get(markets.get(ticker), np.nan),
            "RelativeVolume": vol.tail(5).mean() / avg_vol_60 if avg_vol_60 else np.nan,
        })
    signals = pd.DataFrame(rows).set_index("Ticker")

    # Percentile ranks (0 to 1) make the signals comparable for Zac's scoring
    for column in ["Vol20", "Return20", "RelativeMomentum", "RelativeVolume"]:
        signals[column + "Rank"] = signals[column].rank(pct=True)
    return signals



## running everything

data = load_stock_data(TICKER_FILE)
screening = data["screening"]

# Show what was excluded and why
exclusions = screening.loc[~screening["Eligible"], ["SecurityType", "Exchange", "Currency", "Reason"]]
print(f"\nExcluded {len(exclusions)} tickers:")
display(exclusions)

---

## Part 2 — Client A: Growth strategy (Zac)

**Zac’s code starts here — the following three cells are his contribution.** They compare historical volatility, momentum and KNN, choose a strategy using past tests, and construct `Portfolio_Final_A` with purchase fees and allocation limits.


In [ ]:
# Client A - historical patterns
prices = data["prices_cad"]
markets = screening["Market"].to_dict()
benchmarks = get_benchmarks_cad(prices.index, data["fx"], HISTORY_START, prices.index[-1])
history = {}

for ticker in data["eligible"]:
    p = prices[ticker].dropna()
    v = data["volumes"][ticker].reindex(p.index)
    h = pd.DataFrame(index=p.index)
    h["R5"] = p / p.shift(5) - 1
    h["R20"] = p / p.shift(20) - 1
    h["Vol"] = p.pct_change().rolling(20).std()
    h["Volume"] = v.rolling(5).mean() / v.rolling(60).mean()
    h["Next5"] = p.shift(-5) / p - 1
    history[ticker] = h


def knn(ticker, day):
    h = history[ticker]
    if day not in h.index:
        return np.nan

    i = h.index.get_loc(day)
    cols = ["R5", "R20", "Vol", "Volume"]
    past = h.iloc[:i-4].dropna()
    if len(past) < 30:
        return np.nan

    today = h.loc[day, cols]
    if today.isna().any():
        return np.nan

    scale = past[cols].std().replace(0, 1)
    distance = ((past[cols] - today) / scale) ** 2
    closest = distance.sum(axis=1).nsmallest(20).index
    return past.loc[closest, "Next5"].mean()

In [ ]:
# Score each stock using information available on that date

def stock_scores(day):
    valid = []
    for ticker in data["eligible"]:
        p = prices[ticker].loc[:day].dropna()
        if len(p) >= 65 and day in p.index:
            valid.append(ticker)

    if len(valid) < 10:
        return pd.DataFrame()

    s = calculate_signals(prices[valid], data["volumes"][valid], benchmarks, markets, day)
    s["Sector"] = screening["Sector"]
    sector = s.groupby("Sector")["RelativeMomentum"].transform("mean")

    predictions = []
    for ticker in s.index:
        predictions.append(knn(ticker, day))

    s["SectorRank"] = sector.rank(pct=True)
    s["KNNRank"] = pd.Series(predictions, index=s.index).rank(pct=True).fillna(0.5)
    s["Volatility"] = s["Vol20Rank"]
    s["Momentum"] = (0.5 * s["Return20Rank"] + 0.3 * s["RelativeMomentumRank"]
                     + 0.2 * s["SectorRank"])
    s["KNN"] = (0.4 * s["KNNRank"] + 0.25 * s["Return20Rank"]
                + 0.2 * s["RelativeMomentumRank"] + 0.15 * s["SectorRank"])
    return s


def select_stocks(s, method):
    ranked = s.sort_values(method, ascending=False)

    for n in list(range(12, 21)) + [10, 11]:
        chosen = []
        counts = {}

        for ticker in ranked.index:
            sector = s.loc[ticker, "Sector"]
            if pd.isna(sector) or pd.isna(s.loc[ticker, method]):
                continue
            if counts.get(sector, 0) >= int(n * 0.4):
                continue

            chosen.append(ticker)
            counts[sector] = counts.get(sector, 0) + 1
            if len(chosen) == n:
                break

        if len(chosen) != n:
            continue

        weights = pd.Series(0.5 / n, index=chosen)
        for ticker in chosen:
            sector = s.loc[ticker, "Sector"]
            same = [t for t in chosen if s.loc[t, "Sector"] == sector]
            extra = min(1 - weights.sum(), 0.15 - weights[ticker],
                        0.40 - weights[same].sum())
            if extra > 0:
                weights[ticker] += extra

        if np.isclose(weights.sum(), 1):
            return chosen, weights

    raise ValueError("Could not build a valid Client A portfolio")

In [ ]:
# Compare the 3 strategies on past five-day returns
results = []

for day in prices.index[125:-5:4]:
    s = stock_scores(day)
    if s.empty:
        continue

    row = {"Date": day}
    for method in ["Volatility", "Momentum", "KNN"]:
        try:
            chosen, weights = select_stocks(s, method)
        except ValueError:
            continue

        total = 0
        for ticker in chosen:
            result = history[ticker].loc[day, "Next5"]
            if pd.isna(result):
                break
            total += weights[ticker] * result
        else:
            row[method] = total

    results.append(row)

backtest = pd.DataFrame(results).set_index("Date").dropna()
if len(backtest) < 30:
    raise ValueError("Not enough historical periods to compare strategies")

split = int(len(backtest) * 0.8)
training = backtest.iloc[:split-1]
testing = backtest.iloc[split:]
best = training.mean().idxmax()

print("Historical periods:", len(backtest))
print("Strategy selected:", best)
print("Later test returns (%):")
print((testing.mean() * 100).round(2))

# Construct Client A at the last downloaded closing date
day = prices.index[-1]
s = stock_scores(day)
chosen, weights = select_stocks(s, best)

cad = prices.loc[day, chosen]
local = data["closes_local"].loc[day, chosen]
fx = float(data["fx"].loc[day])
if cad.isna().any() or local.isna().any() or pd.isna(fx) or fx <= 0:
    raise ValueError("Missing closing prices or exchange rate")

budget = CLIENT_BUDGETS["A"]
money = budget - 0.01
for i in range(5):
    shares = money * weights / cad
    fees = np.minimum(2.15, 0.001 * shares).sum() * fx
    money = budget - fees - 0.01

shares = money * weights / cad
fees = np.minimum(2.15, 0.001 * shares).sum() * fx
Portfolio_Final_A = pd.DataFrame({
    "Ticker": chosen,
    "Price": local.values,
    "Currency": screening.loc[chosen, "Currency"].values,
    "Shares": shares.values,
    "Value": (shares * cad).values,
    "Weight": weights.values
}, index=range(1, len(chosen) + 1))

# Quick Client A checks (the team will add final shared validation)
assert 10 <= len(chosen) <= 20
assert set(chosen).issubset(data["eligible"])
assert weights.max() <= 0.1500001
assert weights.groupby(screening.loc[chosen, "Sector"]).sum().max() <= 0.4000001
assert np.isclose(weights.sum(), 1)
assert Portfolio_Final_A["Value"].sum() + fees <= budget

print("Purchase date:", day.date())
display(Portfolio_Final_A)
print("Fees:", round(fees, 2))
print("Cash left:", round(budget - Portfolio_Final_A["Value"].sum() - fees, 2))